In [27]:
import numpy as np
import torch
import torch.nn.functional as F
from monai.config import print_config
from monai.networks.nets import AutoencoderKL, DiffusionModelUNet
from monai.networks.schedulers import DDPMScheduler
from tqdm import tqdm

print_config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MONAI version: 1.5.1
Numpy version: 2.3.5
Pytorch version: 2.9.1
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 9c6d819f97e37f36c72f3bdfad676b455bd2fa0d
MONAI __file__: /home/<username>/projects/ood-detection/.venv/lib/python3.13/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: 5.4.4
Nibabel version: 5.3.2
scikit-image version: 0.25.2
scipy version: 1.16.3
Pillow version: 12.0.0
Tensorboard version: 2.20.0
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.24.1
tqdm version: 4.67.1
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.1.3
pandas version: 2.3.1
einops version: 0.8.1
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependenc

In [28]:
# Significantly reduced capacity to fit in 4GB VRAM
autoencoder = AutoencoderKL(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(32, 64, 64),
    latent_channels=3,
    num_res_blocks=1,
    attention_levels=(False, False, False),
    with_encoder_nonlocal_attn=False,
    with_decoder_nonlocal_attn=False,
).to(device)

In [29]:
from dataloading.load_atlas import load_atlas

train_loader, val_loader, test_loader = load_atlas()

Found 655 image files
Found 655 mask files
Found 300 test image files
Train data: 524 files
Validation data: 131 files


In [4]:
# --- Stage 1: Train AutoencoderKL ---
# Note: In a real scenario, train for significantly more epochs

from monai.inferers import sliding_window_inference

optimizer_ae = torch.optim.Adam(autoencoder.parameters(), lr=1e-4)
recon_loss_fn = torch.nn.MSELoss()
kl_weight = 1e-6

n_epochs_ae = 5  # Reduced for demonstration

print("Starting AutoencoderKL Training...")
autoencoder.train()

for epoch in range(n_epochs_ae):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs_ae}")
    for batch in progress_bar:
        images = batch["image"].to(device)
        images = F.interpolate(images, size=(64, 64, 64), mode='trilinear', align_corners=False)
        optimizer_ae.zero_grad()
        
        reconstruction, z_mu, z_sigma = autoencoder(images)
        
        recons_loss = recon_loss_fn(reconstruction, images)
        kl_loss = 0.5 * torch.sum(z_mu.pow(2) + z_sigma.pow(2) - torch.log(z_sigma.pow(2)) - 1, dim=[1, 2, 3, 4])
        kl_loss = torch.mean(kl_loss)
        
        loss = recons_loss + kl_weight * kl_loss
        loss.backward()
        optimizer_ae.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    print(f"Epoch {epoch+1} finished. Avg Loss: {epoch_loss / len(train_loader):.4f}")

# Save AE weights
torch.save(autoencoder.state_dict(), "autoencoder.pth")

Starting AutoencoderKL Training...


Epoch 1/5: 100%|██████████| 524/524 [01:55<00:00,  4.55it/s, loss=0.0158]


Epoch 1 finished. Avg Loss: 0.0298


Epoch 2/5: 100%|██████████| 524/524 [02:04<00:00,  4.20it/s, loss=0.0131]


Epoch 2 finished. Avg Loss: 0.0175


...

Epoch 3 finished. Avg Loss: 0.0157


Epoch 4/5: 100%|██████████| 524/524 [02:09<00:00,  4.05it/s, loss=0.0107]


Epoch 4 finished. Avg Loss: 0.0147


Epoch 5/5: 100%|██████████| 524/524 [02:10<00:00,  4.01it/s, loss=0.0106] 

Epoch 5 finished. Avg Loss: 0.0139


In [30]:
autoencoder.load_state_dict(torch.load("autoencoder.pth"))

<All keys matched successfully>

In [31]:
# Significantly reduced capacity
unet = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=3,
    out_channels=3,
    channels=(32, 64, 128),
    num_res_blocks=1,
    attention_levels=(False, False, True),
).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000, schedule="linear_beta", beta_start=0.0015, beta_end=0.0195)

In [6]:
# --- Stage 2: Train Diffusion Model ---

optimizer_unet = torch.optim.Adam(unet.parameters(), lr=1e-4)
n_epochs_diff = 10 # Increased epochs
accumulation_steps = 4 # Gradient accumulation for stability

print("Starting Diffusion Model Training...")
autoencoder.eval()
unet.train()

for epoch in range(n_epochs_diff):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs_diff}")
    
    optimizer_unet.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        images = batch["image"].to(device)
        images = F.interpolate(images, size=(64, 64, 64), mode='trilinear', align_corners=False)
        
        # 1. Get Latents
        with torch.no_grad():
            z_mu, z_sigma = autoencoder.encode(images)
            z = autoencoder.sampling(z_mu, z_sigma)
            
        # 2. Add Noise
        noise = torch.randn_like(z).to(device)
        timesteps = torch.randint(0, scheduler.num_train_timesteps, (z.shape[0],), device=device).long()
        noisy_z = scheduler.add_noise(original_samples=z, noise=noise, timesteps=timesteps)
        
        # 3. Denoise
        noise_pred = unet(x=noisy_z, timesteps=timesteps)
        
        loss = F.mse_loss(noise_pred, noise)
        
        # Gradient Accumulation
        loss = loss / accumulation_steps
        loss.backward()
        
        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer_unet.step()
            optimizer_unet.zero_grad()
        
        epoch_loss += loss.item() * accumulation_steps
        progress_bar.set_postfix({"loss": loss.item() * accumulation_steps})

    print(f"Epoch {epoch+1} finished. Avg Loss: {epoch_loss / len(train_loader):.4f}")

# Save Diffusion weights
torch.save(unet.state_dict(), "diffusion_unet.pth")

Starting Diffusion Model Training...


Epoch 1/10: 100%|██████████| 524/524 [01:28<00:00,  5.93it/s, loss=0.222]


Epoch 1 finished. Avg Loss: 0.6207


Epoch 2/10: 100%|██████████| 524/524 [01:33<00:00,  5.58it/s, loss=0.0612]


Epoch 2 finished. Avg Loss: 0.2799


...

Epoch 8 finished. Avg Loss: 0.2292


Epoch 9/10: 100%|██████████| 524/524 [01:33<00:00,  5.63it/s, loss=0.114] 


Epoch 9 finished. Avg Loss: 0.2287


Epoch 10/10: 100%|██████████| 524/524 [01:32<00:00,  5.67it/s, loss=0.0925]


Epoch 10 finished. Avg Loss: 0.2372


In [32]:
unet.load_state_dict(torch.load("diffusion_unet.pth"))

<All keys matched successfully>

In [7]:
from monai.networks.schedulers import PNDMScheduler

def calculate_confidence_score(image, scheduler):
    with torch.no_grad():
        z_mu, z_sigma = autoencoder.encode(image)
        z = autoencoder.sampling(z_mu, z_sigma)

    mse_scores = []
    mse_loss = torch.nn.MSELoss()

    x_0 = image

    for t in scheduler.timesteps:
        with torch.no_grad():
            # Predict noise residual using the U-Net
            # PNDM expects the time 't' to be passed as a tensor
            model_output = unet(z, timesteps=torch.Tensor((t,)).to(device))
            
            # Compute the previous noisy sample (x_t -> x_t-1)
            # The scheduler handles the complex PLMS math internally
            z, _ = scheduler.step(model_output, t, z)
            x = autoencoder.decode(z)
            mse = mse_loss(x_0, x)
            mse_scores.append(mse)
    confidence_score = sum(mse_scores) / len(mse_scores)
    return confidence_score


In [33]:
import torch.nn.functional as F

def calculate_scores(data_loader):
    scheduler = PNDMScheduler(
        num_train_timesteps=1000, 
        beta_start=0.0015, 
        beta_end=0.0205, 
        prediction_type="epsilon"
    )
    scheduler.set_timesteps(num_inference_steps=50)
    scores = []
    for batch in tqdm(data_loader, desc="Calculating Scores"):
        image = batch["image"].to(device)
        if image.ravel().shape[0] == 0:
            print("Empty image encountered, skipping.")
            continue
        image = F.interpolate(image, size=(64, 64, 64), mode='trilinear', align_corners=False)
        score = calculate_confidence_score(image, scheduler)
        scores.append(score.item())
    return scores

In [ ]:
val_scores = calculate_scores(val_loader)

In [42]:
import pickle

with open("ldm_val_scores.pkl", "wb") as f:
    pickle.dump(val_scores, f)

In [34]:
import pickle

with open("ldm_val_scores.pkl", "rb") as f:
    val_scores = pickle.load(f)

In [10]:
threshold = np.percentile(val_scores, 95)

In [11]:
threshold

np.float64(0.36401431262493134)

In [10]:
test_scores = calculate_scores(test_loader)
test_scores[10:]

Calculating Scores:   0%|          | 0/300 [00:00<?, ?it/s]

Calculating Scores: 100%|██████████| 300/300 [16:44<00:00,  3.35s/it]


[0.30450239777565,
 0.29997718334198,
 0.29592934250831604,
 0.31013256311416626,
 0.33257678151130676,
 0.3157682716846466,
 0.30541837215423584,
 0.32113921642303467,
 0.31359922885894775,
 0.3284556269645691,
 0.30097174644470215,
 0.3152461647987366,
 0.3181365430355072,
 0.3239773213863373,
 0.29963263869285583,
 0.3139336109161377,
 0.30423039197921753,
 0.3138567805290222,
 0.32990705966949463,
 0.30456313490867615,
 0.3092450201511383,
 0.30666011571884155,
 0.30647748708724976,
 0.29786616563796997,
 0.29507336020469666,
 0.2990255653858185,
 0.308329701423645,
 0.30980196595191956,
 0.30984365940093994,
 0.3069896399974823,
 0.3207476735115051,
 0.289562463760376,
 0.3097216784954071,
 0.30932125449180603,
 0.32697075605392456,
 0.32278093695640564,
 0.30309775471687317,
 0.2977534532546997,
 0.2940637767314911,
 0.30576133728027344,
 0.2942206859588623,
 0.3303605318069458,
 0.30736497044563293,
 0.32920873165130615,
 0.3328007459640503,
 0.3170635998249054,
 0.3096939325332

In [41]:
import pickle

with open("ldm_test_scores.pkl", "wb") as f:
    pickle.dump(test_scores, f)

In [35]:
import pickle

with open("ldm_test_scores.pkl", "rb") as f:
    test_scores = pickle.load(f)

In [36]:
sub_test_scores = test_scores[:40]

In [36]:
from dataloading.load_chaos import load_chaos

chaos_loader = load_chaos()

Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


In [37]:
chaos_scores = calculate_scores(chaos_loader)
print(chaos_scores[:10])

Calculating Scores: 100%|██████████| 40/40 [02:18<00:00,  3.47s/it]

[0.0894995778799057, 0.24432170391082764, 0.26431116461753845, 0.25973188877105713, 0.272329181432724, 0.25444909930229187, 0.25601014494895935, 0.275253027677536, 0.29174891114234924, 0.27228763699531555]


In [38]:
scores = np.concatenate([np.array(chaos_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(chaos_scores), np.zeros_like(test_scores)])
labels

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [39]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.8483, fpr90: 0.0367, fpr95: 0.0467, fpr99: 0.1733


In [20]:
from sklearn.metrics import accuracy_score

threshold = np.percentile(val_scores, 95)
accuracy_score(labels, scores > threshold)

0.8147058823529412

In [21]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, scores > threshold)
cm

array([[277,  23],
       [ 40,   0]])

In [40]:
from dataloading.load_brats import load_brats
brats_loader = load_brats()

Found 585 Train BRATS volumes
Found 87 Test BRATS volumes


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [41]:
brats_scores = calculate_scores(brats_loader)
print(brats_scores[:10])

Calculating Scores:   0%|          | 0/672 [00:00<?, ?it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00101728

ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545123

Calculating Scores:   0%|          | 1/672 [00:07<1:21:32,  7.29s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00100171

Calculating Scores:   3%|▎         | 22/672 [01:16<35:37,  3.29s/it] WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545999

Calculating Scores:   3%|▎    

Empty image encountered, skipping.


Calculating Scores:  42%|████▏     | 285/672 [16:03<21:56,  3.40s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:9.95169e-05

Calculating Scores:  44%|████▍     | 295/672 [16:37<21:31,  3.42s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000398104

Calculating Scores:  44%|████▍     | 298/672 [16:47<21:21,  3.43s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x560d250ee120): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000136793

Calculating Scores:  45%|████▍     | 301/672 [16:58<21:21,  3.46s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImage

[0.14388170838356018, 0.3262254297733307, 0.26817983388900757, 0.3381716012954712, 0.32499080896377563, 0.3064884841442108, 0.3410870432853699, 0.3193682134151459, 0.30534204840660095, 0.3439984619617462]


In [44]:
import pickle

with open("ldm_brats_scores.pkl", "wb") as f:
    pickle.dump(brats_scores, f)

In [42]:
scores = np.concatenate([np.array(brats_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(brats_scores), np.zeros_like(test_scores)])

In [43]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.8662, fpr90: 0.6400, fpr95: 0.7500, fpr99: 0.8133
